"""
Fetch Real Space-Weather Data (October - November 2026) and Export to Excel
===========================================================================
Sources:
  1. NASA GSFC OMNIWeb / CDAWeb HAPI (DSCOVR/ACE solar wind & geomagnetic indices)
     https://cdaweb.gsfc.nasa.gov/hapi
  2. GFZ Potsdam Geomagnetic Kp Index API (3-hourly planetary Kp, definitive/nowcast)
     https://kp.gfz-potsdam.de/app/json/
  3. NOAA SWPC / NCEI GOES XRS Solar X-Ray Flux & Flare Events
     - Archive (1-min NetCDF): https://data.ngdc.noaa.gov/platforms/solar-space-observing-satellites/goes/
     - Real-time (last 7 days JSON): https://services.swpc.noaa.gov/json/goes/primary/

Requirements:
  pip install requests pandas numpy openpyxl xarray netCDF4
  (xarray + netCDF4 are only needed for full Oct-Nov NCEI GOES NetCDF archives;
   the script still runs without them using the 7-day NOAA SWPC JSON feed.)
"""

In [22]:
from datetime import datetime, timezone
from io import BytesIO, StringIO
import re
import tempfile
import os

import numpy as np
import pandas as pd
import requests
from openpyxl.styles import Alignment, Font, PatternFill

### Configuration


In [23]:

START_UTC = "2026-10-01T00:00:00Z"
END_UTC = "2026-11-30T23:59:59Z"
OMNI_RESOLUTION = "1min"  # Choose "1min" (OMNI_HRO2_1MIN) or "1hr" (OMNI2_H0_MRG1HR)
GOES_SATELLITE = "goes18"  # Operational primary/secondary GOES-R satellite ("goes18" or "goes19")
OUTPUT_EXCEL = "Real_Space_Weather_Oct_Nov_2026.xlsx"
REQUEST_TIMEOUT = 60

### 1. NASA GSFC OMNIWeb / CDAWeb HAPI (Solar Wind & Geomagnetic Indices)

In [24]:
def fetch_nasa_omni_hapi(
    start_utc: str = START_UTC,
    end_utc: str = END_UTC,
    resolution: str = OMNI_RESOLUTION,
) -> pd.DataFrame:
    """Fetch OMNI solar wind data via NASA CDAWeb HAPI REST endpoint."""
    base_url = "https://cdaweb.gsfc.nasa.gov/hapi"

    if resolution == "1min":
        dataset_id = "OMNI_HRO2_1MIN"
        desired_params = ["flow_speed", "BZ_GSM", "BY_GSM", "proton_density", "Pressure", "SYM_H"]
    elif resolution == "1hr":
        dataset_id = "OMNI2_H0_MRG1HR"
        desired_params = ["V", "BZ_GSM", "BY_GSM", "N", "DST1800", "KP1800"]
    else:
        raise ValueError("resolution must be '1min' or '1hr'")

    print(f"[1/4] Querying NASA CDAWeb HAPI metadata for {dataset_id}...")
    info_resp = requests.get(
        f"{base_url}/info",
        params={"id": dataset_id},
        timeout=REQUEST_TIMEOUT,
    )
    info_resp.raise_for_status()
    meta = info_resp.json()

    # Keep only parameters confirmed present in the dataset metadata
    available = {p["name"]: p for p in meta.get("parameters", [])}
    selected = [p for p in desired_params if p in available]
    if not selected:
        raise RuntimeError(f"None of {desired_params} found in {dataset_id} HAPI metadata.")

    # Respect dataset stopDate if the requested end_utc is newer than available OMNI archive
    stop_date = meta.get("stopDate", end_utc)
    effective_end = min(end_utc, stop_date)
    if effective_end < start_utc:
        print(
            f"  [!] {dataset_id} currently ends at {stop_date}, before {start_utc}. "
            "OMNIWeb typically has a 1-2 month processing lag."
        )
        return pd.DataFrame()

    print(f"  Downloading {dataset_id} ({start_utc} to {effective_end})...")
    data_resp = requests.get(
        f"{base_url}/data",
        params={
            "id": dataset_id,
            "parameters": ",".join(selected),
            "time.min": start_utc,
            "time.max": effective_end,
            "format": "csv",
            "include": "header",
        },
        timeout=REQUEST_TIMEOUT * 3,
    )
    data_resp.raise_for_status()

    # Robustly check if the response is actually an error/status report instead of CSV data
    response_text = data_resp.text.strip()
    if "status" in response_text or "\"code\"" in response_text or response_text.startswith("{"):
        print(f"  [!] HAPI Server returned an error or status response:\n{response_text[:300]}...")
        print("  Falling back to an empty DataFrame for OMNI data.")
        return pd.DataFrame()

    df = pd.read_csv(StringIO(response_text))
    if df.empty:
        return df

    time_col = df.columns[0]
    df.rename(columns={time_col: "Datetime_UTC"}, inplace=True)
    df["Datetime_UTC"] = pd.to_datetime(df["Datetime_UTC"], utc=True).dt.tz_localize(None)

    # Replace HAPI-declared fill values (e.g., 99999.9, 9999.99) with NaN
    for param in selected:
        fill_val = available[param].get("fill")
        if fill_val is not None:
            try:
                fill_num = float(fill_val)
                df[param] = pd.to_numeric(df[param], errors="coerce")
                df.loc[np.isclose(df[param], fill_num, rtol=1e-5, atol=1e-5), param] = np.nan
            except (ValueError, TypeError):
                pass

    print(f"  Retrieved {len(df):,} rows from NASA {dataset_id}.")
    return df

### 2. GFZ Potsdam Geomagnetic Kp Index (3-Hourly Official API)

In [25]:
def fetch_gfz_kp(start_utc: str = START_UTC, end_utc: str = END_UTC) -> pd.DataFrame:
    """Fetch official 3-hourly Kp index from GFZ Potsdam JSON API."""
    url = "https://kp.gfz-potsdam.de/app/json/"
    print(f"[2/4] Fetching GFZ Potsdam Kp index ({start_utc} to {end_utc})...")
    resp = requests.get(
        url,
        params={"start": start_utc, "end": end_utc, "index": "Kp"},
        timeout=REQUEST_TIMEOUT,
    )
    resp.raise_for_status()
    payload = resp.json()

    # The GFZ JSON payload includes a 'meta' dict alongside parallel lists
    df = pd.DataFrame(
        {
            "Datetime_UTC": pd.to_datetime(payload["datetime"], utc=True).tz_localize(None),
            "Kp_Index": pd.to_numeric(payload["Kp"], errors="coerce"),
            "Status": payload.get("status", ["unknown"] * len(payload["datetime"])),
        }
    )
    print(f"  Retrieved {len(df):,} 3-hourly Kp records from GFZ Potsdam.")
    return df

### 3. NOAA NCEI GOES XRS 1-Minute Archive (Full October - November)

In [26]:
def fetch_noaa_goes_xrs_archive(
    start_utc: str = START_UTC,
    end_utc: str = END_UTC,
    satellite: str = GOES_SATELLITE,
) -> pd.DataFrame:
    """
    Download daily 1-minute GOES XRS NetCDF-4 files from NOAA NCEI for full
    multi-month coverage (October & November). Requires xarray + netCDF4.
    """
    try:
        import xarray as xr
    except ImportError:
        print(
            "[3/4] Skipping NOAA NCEI NetCDF archive (install 'xarray' and 'netCDF4' "
            "for full multi-month GOES XRS data). Will still fetch 7-day SWPC JSON."
        )
        return pd.DataFrame()

    start_dt = pd.to_datetime(start_utc).tz_localize(None)
    end_dt = pd.to_datetime(end_utc).tz_localize(None)
    months = pd.period_range(start=start_dt, end=end_dt, freq="M")

    # Operational L2 1-min average folder (updated daily); science folder may lag
    base_dir = (
        f"https://data.ngdc.noaa.gov/platforms/solar-space-observing-satellites/"
        f"goes/{satellite}/l2/data/xrsf-l2-avg1m"
    )
    frames = []
    print(f"[3/4] Checking NOAA NCEI {satellite.upper()} 1-min XRS archive...")

    for period in months:
        month_url = f"{base_dir}/{period.year}/{period.month:02d}/"
        resp = requests.get(month_url, timeout=REQUEST_TIMEOUT)
        if resp.status_code != 200:
            print(f"  [!] Directory not available: {month_url} (HTTP {resp.status_code})")
            continue

        nc_files = sorted(set(re.findall(r'href="(dn_xrsf-l2-avg1m_[^"]+\.nc)"', resp.text)))
        print(f"  Found {len(nc_files)} daily NetCDF files in {period.year}-{period.month:02d}.")

        for fname in nc_files:
            file_url = month_url + fname
            try:
                f_resp = requests.get(file_url, timeout=REQUEST_TIMEOUT)
                f_resp.raise_for_status()
                with tempfile.NamedTemporaryFile(suffix=".nc", delete=False) as tmp:
                    tmp.write(f_resp.content)
                    tmp_path = tmp.name
                try:
                    with xr.open_dataset(tmp_path) as ds:
                        day_df = pd.DataFrame(
                            {
                                "Datetime_UTC": pd.to_datetime(ds["time"].values),
                                "XRS_A_0.05_0.4nm_Wm2": ds["xrsa_flux"].values,
                                "XRS_B_0.1_0.8nm_Wm2": ds["xrsb_flux"].values,
                            }
                        )
                        frames.append(day_df)
                finally:
                    if os.path.exists(tmp_path):
                        os.remove(tmp_path)
            except Exception as exc:
                print(f"  [!] Failed to read {fname}: {exc}")

    if not frames:
        return pd.DataFrame()

    df = pd.concat(frames, ignore_index=True)
    df = df[(df["Datetime_UTC"] >= start_dt) & (df["Datetime_UTC"] <= end_dt)]
    df.sort_values("Datetime_UTC", inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f"  Retrieved {len(df):,} 1-minute GOES XRS rows from NOAA NCEI.")
    return df

### 4. NOAA SWPC Real-Time GOES XRS & Flare Catalog (Last 7 Days JSON)

In [27]:
def fetch_noaa_swpc_xrs_realtime() -> tuple[pd.DataFrame, pd.DataFrame]:
    """Fetch last 7 days of real-time GOES XRS flux and flare list from NOAA SWPC."""
    xrs_url = "https://services.swpc.noaa.gov/json/goes/primary/xrays-7-day.json"
    flare_url = "https://services.swpc.noaa.gov/json/goes/primary/xray-flares-7-day.json"
    print("[4/4] Fetching NOAA SWPC real-time GOES XRS (last 7 days) & flare events...")

    # 1-minute X-ray flux
    xrs_raw = pd.read_json(xrs_url)
    xrs_df = xrs_raw.pivot(index="time_tag", columns="energy", values="flux").reset_index()
    xrs_df.rename(
        columns={
            "time_tag": "Datetime_UTC",
            "0.05-0.4nm": "XRS_A_0.05_0.4nm_Wm2",
            "0.1-0.8nm": "XRS_B_0.1_0.8nm_Wm2",
        },
        inplace=True,
    )
    xrs_df["Datetime_UTC"] = pd.to_datetime(xrs_df["Datetime_UTC"], utc=True).dt.tz_localize(None)
    xrs_df.sort_values("Datetime_UTC", inplace=True)

    # Flare events
    try:
        flares_df = pd.read_json(flare_url)
        for col in ["begin_time", "max_time", "end_time"]:
            if col in flares_df.columns:
                flares_df[col] = pd.to_datetime(
                    flares_df[col], utc=True, errors="coerce"
                ).dt.tz_localize(None)
    except Exception as exc:
        print(f"  [!] Could not load 7-day flare events JSON: {exc}")
        flares_df = pd.DataFrame()

    print(f"  Retrieved {len(xrs_df):,} real-time XRS rows and {len(flares_df):,} flare events.")
    return xrs_df, flares_df

### Build Hourly Merged Table (UTC + IST for Geiger-Muller Alignment)

In [28]:
def build_hourly_merged(
    omni_df: pd.DataFrame, kp_df: pd.DataFrame, xrs_df: pd.DataFrame
) -> pd.DataFrame:
    """Resample all available datasets to a common 1-hour UTC grid (+ IST column)."""
    grid = pd.date_range(
        start=pd.to_datetime(START_UTC).tz_localize(None),
        end=pd.to_datetime(END_UTC).tz_localize(None),
        freq="1h",
    )
    merged = pd.DataFrame({"Datetime_UTC": grid})
    merged["Datetime_IST"] = merged["Datetime_UTC"] + pd.Timedelta(hours=5, minutes=30)

    if not omni_df.empty:
        omni_hr = (
            omni_df.set_index("Datetime_UTC")
            .resample("1h")
            .mean(numeric_only=True)
            .reset_index()
        )
        merged = merged.merge(omni_hr, on="Datetime_UTC", how="left")

    if not kp_df.empty:
        # Forward-fill each 3-hourly Kp value across its 3-hour validity window
        kp_hr = (
            kp_df.set_index("Datetime_UTC")[["Kp_Index"]]
            .resample("1h")
            .ffill(limit=2)
            .reset_index()
        )
        merged = merged.merge(kp_hr, on="Datetime_UTC", how="left")

    if not xrs_df.empty:
        xrs_hr = (
            xrs_df.set_index("Datetime_UTC")
            .resample("1h")
            .max(numeric_only=True)  # Hourly peak X-ray flux preserves flare peaks
            .reset_index()
        )
        merged = merged.merge(xrs_hr, on="Datetime_UTC", how="left")

    return merged

### Styled Excel Export

In [29]:
def style_worksheet(ws, df: pd.DataFrame) -> None:
    """Apply clean header styling, column widths, freeze panes, and filters."""
    header_fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    header_font = Font(color="FFFFFF", bold=True)
    center_align = Alignment(horizontal="center", vertical="center")

    for col_idx, col_name in enumerate(df.columns, 1):
        cell = ws.cell(row=1, column=col_idx)
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = center_align
        col_letter = cell.column_letter
        ws.column_dimensions[col_letter].width = max(len(str(col_name)) + 4, 18)

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions


def save_to_excel(
    output_path: str,
    omni_df: pd.DataFrame,
    kp_df: pd.DataFrame,
    xrs_df: pd.DataFrame,
    flares_df: pd.DataFrame,
    merged_df: pd.DataFrame,
) -> None:
    """Write all fetched datasets and source provenance metadata to Excel."""
    metadata_df = pd.DataFrame(
        [
            {
                "Dataset": "NASA GSFC OMNI Solar Wind",
                "Endpoint": "https://cdaweb.gsfc.nasa.gov/hapi",
                "Resolution": OMNI_RESOLUTION,
                "Rows_Retrieved": len(omni_df),
                "Citation": "King & Papitashvili (2005), NASA/GSFC SPDF OMNIWeb",
            },
            {
                "Dataset": "GFZ Potsdam Kp Index",
                "Endpoint": "https://kp.gfz-potsdam.de/app/json/",
                "Resolution": "3-hourly",
                "Rows_Retrieved": len(kp_df),
                "Citation": "Matzka et al. (2021), GFZ Potsdam, doi:10.5880/Kp.0001",
            },
            {
                "Dataset": "NOAA NCEI / SWPC GOES XRS",
                "Endpoint": "https://data.ngdc.noaa.gov/ & https://services.swpc.noaa.gov/json/goes/primary/",
                "Resolution": "1-minute",
                "Rows_Retrieved": len(xrs_df),
                "Citation": "NOAA NCEI & SWPC GOES-R EXIS/XRS Level 2",
            },
            {
                "Dataset": "Retrieval Timestamp (UTC)",
                "Endpoint": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC"),
                "Resolution": "N/A",
                "Rows_Retrieved": len(merged_df),
                "Citation": f"Window: {START_UTC} to {END_UTC}",
            },
        ]
    )

    sheets = [
        ("Merged_Hourly_UTC_IST", merged_df),
        ("GFZ_Kp_3Hourly", kp_df),
        ("NASA_OMNI_SolarWind", omni_df),
        ("NOAA_GOES_XRS_1Min", xrs_df),
        ("NOAA_SWPC_Flares_7Day", flares_df),
        ("Data_Sources_Metadata", metadata_df),
    ]

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for sheet_name, df in sheets:
            if df.empty:
                df = pd.DataFrame({"Notice": ["No records returned for this window/endpoint."]})
            df.to_excel(writer, index=False, sheet_name=sheet_name)
            style_worksheet(writer.sheets[sheet_name], df)

    print(f"\nSaved workbook successfully: {output_path}")


### Main Execution

In [30]:
def main() -> None:
    omni_df = fetch_nasa_omni_hapi()
    kp_df = fetch_gfz_kp()
    xrs_archive_df = fetch_noaa_goes_xrs_archive()
    xrs_rt_df, flares_df = fetch_noaa_swpc_xrs_realtime()

    # Combine NCEI historical XRS with real-time 7-day SWPC JSON (deduplicating overlap)
    if not xrs_archive_df.empty and not xrs_rt_df.empty:
        xrs_combined = (
            pd.concat([xrs_archive_df, xrs_rt_df], ignore_index=True)
            .drop_duplicates(subset=["Datetime_UTC"], keep="first")
            .sort_values("Datetime_UTC")
            .reset_index(drop=True)
        )
    elif not xrs_archive_df.empty:
        xrs_combined = xrs_archive_df
    else:
        xrs_combined = xrs_rt_df

    merged_df = build_hourly_merged(omni_df, kp_df, xrs_combined)

    # Helper to clean up any timezone-aware datetime columns to avoid Excel export errors
    def remove_timezone(df: pd.DataFrame) -> pd.DataFrame:
        if df.empty:
            return df
        df = df.copy()
        for col in df.columns:
            if pd.api.types.is_datetime64_any_dtype(df[col]):
                try:
                    df[col] = df[col].dt.tz_localize(None)
                except TypeError:
                    # Already timezone-naive or conversion handled
                    df[col] = df[col].dt.tz_convert(None).dt.tz_localize(None)
        return df

    # Ensure all dataframes are timezone naive before exporting to Excel
    omni_df = remove_timezone(omni_df)
    kp_df = remove_timezone(kp_df)
    xrs_combined = remove_timezone(xrs_combined)
    flares_df = remove_timezone(flares_df)
    merged_df = remove_timezone(merged_df)

    save_to_excel(OUTPUT_EXCEL, omni_df, kp_df, xrs_combined, flares_df, merged_df)


if __name__ == "__main__":
    main()

[1/4] Querying NASA CDAWeb HAPI metadata for OMNI_HRO2_1MIN...
  [!] HAPI Server returned an error or status response:
,
"status": {"code": 1411, "message": "HAPI error 1411: Parameter out of order"}
}...
  Falling back to an empty DataFrame for OMNI data.
[2/4] Fetching GFZ Potsdam Kp index (2026-10-01T00:00:00Z to 2026-11-30T23:59:59Z)...
  Retrieved 412 3-hourly Kp records from GFZ Potsdam.
[3/4] Checking NOAA NCEI GOES18 1-min XRS archive...
  Found 31 daily NetCDF files in 2026-10.
  Found 20 daily NetCDF files in 2026-11.
  Retrieved 73,440 1-minute GOES XRS rows from NOAA NCEI.
[4/4] Fetching NOAA SWPC real-time GOES XRS (last 7 days) & flare events...
  Retrieved 10,072 real-time XRS rows and 79 flare events.

Saved workbook successfully: Real_Space_Weather_Oct_Nov_2026.xlsx
